# Cell 1

In [ ]:
# =============================================================================
# Thesis — GeoAI Café Site Selection · Milan
# Phase 2 · Notebook 8 · GeoJSON Export and WebGIS Preparation
# =============================================================================
#
# Purpose:
#   Assemble the single, flat, production-ready GeoJSON file that the
#   Phase 3 WebGIS frontend (Mapbox GL JS / Leaflet / NASA WorldWind)
#   will consume directly.
#
#   This notebook does NOT perform any new analysis. It is a data
#   engineering step: it collects, cleans, and structures all Gold layer
#   outputs from Notebooks 4–7 into one coherent deliverable and
#   validates it is frontend-ready.
#
# Design principles for the output GeoJSON:
#
#   1. Single flat file — no joins required on the frontend.
#      All scores, SHAP values, labels, and display attributes
#      are columns in the same FeatureCollection.
#
#   2. WGS84 (EPSG:4326) — GeoJSON standard. H3 polygon geometries
#      in WGS84 so they render correctly in any web mapping library.
#
#   3. Human-readable column names — the frontend reads these directly
#      into popups and tooltips. Raw column names like
#      'metro_count_norm' become 'Metro Access' in the display layer.
#
#   4. Suitability tier classification — a categorical field
#      ('suitability_tier') is added based on both AHP score and
#      RF probability, providing a simple traffic-light signal for
#      non-technical entrepreneurs.
#
#   5. Minimal file size — only columns needed by the frontend are
#      exported. Raw normalized features are omitted (available in
#      gold_features_normalized.geojson if needed).
#
# Suitability tier logic (quantile-derived from training cells only):
#   HIGH   — combined_score >= q_0.67 of training-cell distribution
#   MEDIUM — q_0.33 <= combined_score < q_0.67
#   LOW    — combined_score < q_0.33 (neither threshold met)
#   Thresholds are computed from training cells only to prevent validation cell leakage.
#
# Inputs (all from Gold layer):
#   - gold_shap_scores.geojson             — per-cell SHAP vectors + all scores
#   - gold_features_normalized.geojson     — h3_id cross-join validation only
#   - gold_ahp_scores.geojson              — h3_id cross-join validation only
#   - gold_rf_scores.geojson               — h3_id cross-join validation only
#   - gold_ahp_vs_shap_comparison.csv      — feature display names and cluster map
#   - gold_train_val_idx.pkl               — train/val split indices (tier thresholds)
#   - gold_performance.json                — RF model performance metrics
#   - gold_feature_cols.pkl                — feature column list (SHAP count validation)
#
# Outputs:
#   - gold_webgis_layer.geojson   — production WebGIS layer (primary output)
#   - gold_webgis_summary.json    — metadata / legend config for frontend
#   - gold_nb8_log.txt
#
# CRS: EPSG:4326 throughout (GeoJSON standard)
# =============================================================================

#Cell 2

In [ ]:
!pip install geopandas -q
!pip install pandas -q
!pip install numpy -q

import os
import json
import datetime
import warnings
import pickle
warnings.filterwarnings('ignore')
os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

import numpy as np
import pandas as pd
import geopandas as gpd

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Thesis'
BRONZE_PATH  = os.path.join(PROJECT_ROOT, 'Bronze')
SILVER_PATH  = os.path.join(PROJECT_ROOT, 'Silver')
GOLD_PATH    = os.path.join(PROJECT_ROOT, 'Gold')

WGS84_CRS    = 'EPSG:4326'

print(f"Notebook 8 started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Purpose: Assemble production WebGIS GeoJSON from Gold layer outputs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook 8 started: 2026-04-26 13:13
Purpose: Assemble production WebGIS GeoJSON from Gold layer outputs


# Cell 3

In [ ]:
# =============================================================================
# LOAD GOLD LAYER INPUTS
# =============================================================================

print("Loading Gold layer inputs...")
print()

# Primary source: SHAP scores GeoJSON from Notebook 7
# Contains: h3_id, label, cafe_count, ahp_score, ahp_rank,
#           rf_probability, rf_rank, rank_diff, shap_* columns, geometry
shap_gdf = gpd.read_file(
    os.path.join(GOLD_PATH, 'gold_shap_scores.geojson')
)
print(f"SHAP scores GeoJSON:  {len(shap_gdf)} cells, "
      f"{len(shap_gdf.columns)} columns")
print(f"  CRS: {shap_gdf.crs}")

# AHP vs SHAP comparison table — provides cluster assignments
# and human-readable feature names used in the display layer
comparison_df = pd.read_csv(
    os.path.join(GOLD_PATH, 'gold_ahp_vs_shap_comparison.csv')
)
print(f"AHP vs SHAP table:    {len(comparison_df)} features")

# Performance metrics — dynamically loaded from NB6 export
# (loaded in Cell 10 where it is consumed)

# Ensure SHAP GeoJSON is in WGS84
if shap_gdf.crs and shap_gdf.crs.to_epsg() != 4326:
    shap_gdf = shap_gdf.to_crs(WGS84_CRS)

print()
print("All inputs loaded and confirmed in WGS84")

Loading Gold layer inputs...

SHAP scores GeoJSON:  971 cells, 24 columns
  CRS: EPSG:4326
AHP vs SHAP table:    13 features

All inputs loaded and confirmed in WGS84


#Cell 3b

In [ ]:
# =============================================================================
# CELL 3b — H3_ID CROSS-JOIN VALIDATION ACROSS GOLD FILES
# =============================================================================
#
# The handover spec requires that h3_id sets across all four Gold GeoJSONs
# are identical before the WebGIS merge. If any file has missing or extra
# cells the merged layer will silently drop rows or produce NaN scores.
# This cell loads the four files, compares their h3_id sets, and aborts
# with a clear diagnostic if they diverge.

print("Validating h3_id consistency across Gold GeoJSON files...")
print()

_gold_files_to_check = {
    'gold_shap_scores.geojson':         os.path.join(GOLD_PATH, 'gold_shap_scores.geojson'),
    'gold_features_normalized.geojson': os.path.join(GOLD_PATH, 'gold_features_normalized.geojson'),
    'gold_ahp_scores.geojson':          os.path.join(GOLD_PATH, 'gold_ahp_scores.geojson'),
    'gold_rf_scores.geojson':           os.path.join(GOLD_PATH, 'gold_rf_scores.geojson'),
}

_h3_id_sets = {}
for _fname, _fpath in _gold_files_to_check.items():
    if os.path.exists(_fpath):
        _gdf_tmp = gpd.read_file(_fpath)
        _h3_id_sets[_fname] = set(_gdf_tmp['h3_id'].astype(str))
        print(f"  {_fname:<45} {len(_h3_id_sets[_fname])} cells")
    else:
        print(f"  WARNING: {_fname} not found — skipping from cross-join check.")

if len(_h3_id_sets) >= 2:
    _reference_name  = list(_h3_id_sets.keys())[0]
    _reference_ids   = _h3_id_sets[_reference_name]
    _cross_join_ok   = True
    for _fname, _ids in _h3_id_sets.items():
        if _fname == _reference_name:
            continue
        _only_in_ref   = _reference_ids - _ids
        _only_in_other = _ids - _reference_ids
        if _only_in_ref or _only_in_other:
            _cross_join_ok = False
            print(f"  [FAIL] {_fname} h3_id mismatch vs {_reference_name}:")
            if _only_in_ref:
                print(f"    In {_reference_name} but NOT in {_fname} "
                      f"({len(_only_in_ref)} cells): {sorted(_only_in_ref)[:5]}{'...' if len(_only_in_ref)>5 else ''}")
            if _only_in_other:
                print(f"    In {_fname} but NOT in {_reference_name} "
                      f"({len(_only_in_other)} cells): {sorted(_only_in_other)[:5]}{'...' if len(_only_in_other)>5 else ''}")
        else:
            print(f"  [PASS] {_fname} h3_id set matches {_reference_name}")

    assert _cross_join_ok, (
        "h3_id sets differ across Gold GeoJSON files. "
        "Re-run NB4–NB7 to regenerate consistent Gold outputs before proceeding."
    )
    print()
    print(f"h3_id cross-join validation PASSED — all {len(_h3_id_sets)} files share "
          f"identical h3_id sets ({len(_reference_ids)} cells).")
else:
    print("  WARNING: Fewer than 2 Gold files found — cross-join check skipped.")
print()

Validating h3_id consistency across Gold GeoJSON files...

  gold_shap_scores.geojson                      971 cells
  gold_features_normalized.geojson              971 cells
  gold_ahp_scores.geojson                       971 cells
  gold_rf_scores.geojson                        971 cells
  [PASS] gold_features_normalized.geojson h3_id set matches gold_shap_scores.geojson
  [PASS] gold_ahp_scores.geojson h3_id set matches gold_shap_scores.geojson
  [PASS] gold_rf_scores.geojson h3_id set matches gold_shap_scores.geojson

h3_id cross-join validation PASSED — all 4 files share identical h3_id sets (971 cells).



# Cell 4

In [ ]:
# =============================================================================
# EXTRACT CENTROID COORDINATES FROM EXISTING GEOMETRIES
# =============================================================================
#
# The geometries in gold_shap_scores.geojson are the H3 cell polygons
# already stored in WGS84 from Notebook 7. We use them directly —
# no reconstruction needed.
#
# Centroid lng/lat are added as flat numeric columns for WebGIS tooltip
# placement (Mapbox/Leaflet popup anchors).

print("Extracting centroid coordinates from existing H3 geometries...")

# Confirm geometry column is valid before proceeding
n_invalid = shap_gdf.geometry.isna().sum() + (~shap_gdf.geometry.is_valid).sum()
if n_invalid > 0:
    print(f"  WARNING: {n_invalid} invalid/null geometries found in source GeoJSON")
else:
    print(f"  All {len(shap_gdf)} source geometries are valid")

shap_gdf['centroid_lng'] = shap_gdf.geometry.centroid.x.round(6)
shap_gdf['centroid_lat'] = shap_gdf.geometry.centroid.y.round(6)

print(f"  Centroid coordinates added (lng, lat in WGS84)")
print(f"  Bounding box: "
      f"lng [{shap_gdf['centroid_lng'].min():.4f}, "
      f"{shap_gdf['centroid_lng'].max():.4f}]  "
      f"lat [{shap_gdf['centroid_lat'].min():.4f}, "
      f"{shap_gdf['centroid_lat'].max():.4f}]")

Extracting centroid coordinates from existing H3 geometries...
  All 971 source geometries are valid
  Centroid coordinates added (lng, lat in WGS84)
  Bounding box: lng [9.0683, 9.2683]  lat [45.4005, 45.5337]


# Cell 5

In [ ]:
# =============================================================================
# CELL 5 — SUITABILITY TIER CLASSIFICATION
# =============================================================================
print("Computing suitability tier classification (train-only thresholds)...")
print()

# Load train indices saved by NB6
train_val_idx_path = os.path.join(GOLD_PATH, 'gold_train_val_idx.pkl')
if os.path.exists(train_val_idx_path):
    with open(train_val_idx_path, 'rb') as _f:
        train_val_idx = pickle.load(_f)

    print(f"  gold_train_val_idx.pkl type: {type(train_val_idx)}")
    if isinstance(train_val_idx, dict):
        print(f"  Keys found: {list(train_val_idx.keys())}")
        if 'train' in train_val_idx and 'val' in train_val_idx:
            train_idx = train_val_idx['train']
            val_idx   = train_val_idx['val']
        elif 'train_idx' in train_val_idx and 'val_idx' in train_val_idx:
            train_idx = train_val_idx['train_idx']
            val_idx   = train_val_idx['val_idx']
        else:
            print(f"  WARNING: Unrecognised keys {list(train_val_idx.keys())} "
                  f"— falling back to all cells.")
            train_val_idx = None
    elif isinstance(train_val_idx, (list, np.ndarray)):
        train_idx = list(train_val_idx)
        val_idx   = []
        print(f"  pkl is a flat array — treating all {len(train_idx)} entries as train_idx")
    elif isinstance(train_val_idx, tuple) and len(train_val_idx) == 2:
        train_idx, val_idx = train_val_idx
        print(f"  pkl is a 2-tuple: train={len(train_idx)}, val={len(val_idx)}")
    else:
        print(f"  WARNING: Unrecognised pkl structure ({type(train_val_idx)}) "
              f"— falling back to all cells.")
        train_val_idx = None

    if train_val_idx is not None:
        train_cells = shap_gdf.iloc[train_idx]
        thresholds_source = "training cells only (held-out thresholds)"
        print(f"  Loaded train_val_idx: {len(train_idx)} train, {len(val_idx)} val cells")
    else:
        train_cells = shap_gdf
        thresholds_source = "ALL cells (fallback — unrecognised pkl structure)"
else:
    print(f"  WARNING: gold_train_val_idx.pkl not found — falling back to all cells.")
    print(f"  Re-run NB6 to generate gold_train_val_idx.pkl before final thesis run.")
    train_cells = shap_gdf
    thresholds_source = "ALL cells (fallback — gold_train_val_idx.pkl missing)"

# Combined score via percentile-rank average — computed FIRST so quantile
# thresholds are derived from combined_score on training cells, matching
# the master handover spec (tertile split on combined_score).
n_cells  = len(shap_gdf)
ahp_pct  = shap_gdf['ahp_score'].rank(method='average') / n_cells
rf_pct   = shap_gdf['rf_probability'].rank(method='average') / n_cells
shap_gdf['combined_score'] = ((ahp_pct + rf_pct) / 2).round(4)

# Re-index train_cells to reflect combined_score now added to shap_gdf
if os.path.exists(train_val_idx_path) and train_val_idx is not None:
    train_cells = shap_gdf.iloc[train_idx]

# Compute tier thresholds from combined_score of training cells only
COMBINED_HIGH_THRESHOLD = float(train_cells['combined_score'].quantile(0.67))
COMBINED_LOW_THRESHOLD  = float(train_cells['combined_score'].quantile(0.33))

# Retain AHP/RF threshold variables for log file compatibility (informational only)
AHP_HIGH_THRESHOLD = float(train_cells['ahp_score'].quantile(0.67))
AHP_MED_THRESHOLD  = float(train_cells['ahp_score'].quantile(0.33))
RF_HIGH_THRESHOLD  = float(train_cells['rf_probability'].quantile(0.67))
RF_MED_THRESHOLD   = float(train_cells['rf_probability'].quantile(0.33))

print(f"  Threshold source: {thresholds_source}")
print(f"  Combined score thresholds (q from train cells):")
print(f"    HIGH (q_0.67): {COMBINED_HIGH_THRESHOLD:.4f}")
print(f"    LOW  (q_0.33): {COMBINED_LOW_THRESHOLD:.4f}")
print()

def classify_tier(combined_score):
    """
    Tier classification on combined_score (percentile-rank average of AHP + RF).
    HIGH   — combined_score >= q_0.67 of training-cell distribution (top third)
    MEDIUM — q_0.33 <= combined_score < q_0.67 (middle third)
    LOW    — combined_score < q_0.33 (bottom third)
    Thresholds computed from training cells only to prevent validation leakage.
    """
    if combined_score >= COMBINED_HIGH_THRESHOLD:
        return 'HIGH'
    elif combined_score >= COMBINED_LOW_THRESHOLD:
        return 'MEDIUM'
    else:
        return 'LOW'

shap_gdf['suitability_tier'] = shap_gdf['combined_score'].apply(classify_tier)

tier_counts = shap_gdf['suitability_tier'].value_counts()
print("Suitability tier distribution (combined_score tertiles, thresholds from training cells):")
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    n   = tier_counts.get(tier, 0)
    pct = n / len(shap_gdf) * 100
    print(f"  {tier:<8}: {n:>3} cells ({pct:.1f}%)")
print()

print("Tier vs ground-truth label cross-tabulation (validation-clean):")
crosstab = pd.crosstab(
    shap_gdf['suitability_tier'],
    shap_gdf['label'],
    margins=False
)
crosstab.columns = ['label=0 (neg)', 'label=1 (pos)']
print(crosstab.to_string())
print()
print("  Note: Thresholds derived from training cells only — tier assignment")
print("  of validation cells is not contaminated by their own score distribution.")
print("  HIGH tier cells should be enriched in label=1.")
print("  LOW tier cells should be enriched in label=0.")
print("  Mismatches are the thesis's analytically interesting divergences.")

print()
print("Combined suitability score (percentile-rank average of AHP and RF):")
print(f"  Mean:  {shap_gdf['combined_score'].mean():.4f}  (expected ~0.50)")
print(f"  Min:   {shap_gdf['combined_score'].min():.4f}")
print(f"  Max:   {shap_gdf['combined_score'].max():.4f}")

Computing suitability tier classification (train-only thresholds)...

  gold_train_val_idx.pkl type: <class 'dict'>
  Keys found: ['train_idx', 'val_idx']
  Loaded train_val_idx: 655 train, 316 val cells
  Threshold source: training cells only (held-out thresholds)
  Combined score thresholds (q from train cells):
    HIGH (q_0.67): 0.6153
    LOW  (q_0.33): 0.3048

Suitability tier distribution (combined_score tertiles, thresholds from training cells):
  HIGH    : 359 cells (37.0%)
  MEDIUM  : 324 cells (33.4%)
  LOW     : 288 cells (29.7%)

Tier vs ground-truth label cross-tabulation (validation-clean):
                  label=0 (neg)  label=1 (pos)
suitability_tier                              
HIGH                         83            276
LOW                         288              0
MEDIUM                      255             69

  Note: Thresholds derived from training cells only — tier assignment
  of validation cells is not contaminated by their own score distribution.
  HIGH

# Cell 6

In [ ]:
# =============================================================================
# ADD DISPLAY-FRIENDLY SHAP ATTRIBUTION COLUMNS
# =============================================================================
#
# The WebGIS tooltip shows the top-N features driving each cell's score.
# Rather than displaying raw column names and SHAP floats, we:
#   1. Rename SHAP columns to human-readable display names
#   2. Add a 'top3_factors' string column — the three features with the
#      largest absolute SHAP values for that cell, with direction
#      (positive = boosts suitability, negative = reduces it)
#
# This allows the WebGIS popup to show something like:
#   "Key factors: ↑ Metro Access, ↑ Retail Density, ↓ Competitor Saturation"
# without the frontend needing to parse SHAP arrays.

print("Building display-friendly SHAP attribution columns...")
print()

# Map from raw shap column name to display name
SHAP_DISPLAY_MAP = {
    'shap_metro_count_norm':               'Metro Access',
    'shap_transit_density_norm':           'Bus/Tram Density',
    'shap_network_centrality_norm':        'Network Centrality',
    'shap_pop_density_norm':               'Population Density',
    'shap_office_density_norm':            'Office Density',
    'shap_university_proximity_norm':      'University Proximity',
    'shap_night_light_norm':               'Night Light',
    'shap_retail_density_norm':            'Retail Density',
    'shap_tourist_poi_count_norm':         'Tourist POIs',
    'shap_poi_diversity_norm':             'POI Diversity',
    'shap_local_cafe_density_norm':        'Café Density',
    'shap_competitor_saturation_inv_norm': 'Market Opportunity',
    'shap_pedestrian_street':              'Pedestrian Street',
}
# Note: competitor_saturation is inverted (high = low saturation = good).
# Renamed to 'Market Opportunity' in display to align directionality:
# positive SHAP on this feature means high market opportunity (low saturation).

def build_top3_factors(row):
    """
    Returns a pipe-separated string of the top 3 SHAP drivers for a cell.
    Format: 'FeatureName (direction)|FeatureName (direction)|...'
    Arrow up (↑) = feature increases predicted suitability
    Arrow down (↓) = feature decreases predicted suitability
    """
    # Get current SHAP columns (pre‑rename names) dynamically at call time
    shap_cols = [c for c in row.index if c.startswith('shap_') and c != 'shap_base_value']
    shap_vals = {col: row[col] for col in shap_cols if col in row.index}
    # Sort by absolute SHAP descending
    sorted_feats = sorted(shap_vals.items(), key=lambda x: abs(x[1]), reverse=True)
    top3 = sorted_feats[:3]
    parts = []
    for col, val in top3:
        display = SHAP_DISPLAY_MAP.get(col, col.replace('shap_', ''))
        arrow   = '↑' if val >= 0 else '↓'
        parts.append(f"{arrow} {display}")
    return ' | '.join(parts)

shap_gdf['top3_factors'] = shap_gdf.apply(build_top3_factors, axis=1)

print("Sample top3_factors values:")
sample = shap_gdf[['h3_id', 'suitability_tier', 'top3_factors']].head(5)
for _, row in sample.iterrows():
    print(f"  [{row['suitability_tier']}] {row['top3_factors']}")

print()

# Also rename SHAP columns to display names for the WebGIS export
# This makes them self-documenting in the GeoJSON properties
shap_rename_map = {
    col: f"shap_{SHAP_DISPLAY_MAP[col].replace('/', '_').replace(' ', '_')}"
    for col in shap_gdf.columns if col.startswith('shap_') and col != 'shap_base_value' and col in SHAP_DISPLAY_MAP
}
shap_gdf = shap_gdf.rename(columns=shap_rename_map)

print(f"SHAP columns renamed to display names ({len(shap_rename_map)} columns)")

Building display-friendly SHAP attribution columns...

Sample top3_factors values:
  [LOW] ↓ Café Density | ↓ POI Diversity | ↓ Market Opportunity
  [LOW] ↓ Café Density | ↓ POI Diversity | ↓ Market Opportunity
  [MEDIUM] ↑ Population Density | ↓ POI Diversity | ↑ Market Opportunity
  [LOW] ↓ POI Diversity | ↓ Café Density | ↑ Bus/Tram Density
  [LOW] ↓ Café Density | ↓ POI Diversity | ↓ Market Opportunity

SHAP columns renamed to display names (13 columns)


# Cell 7

In [ ]:
# =============================================================================
# ASSEMBLE THE FINAL WEBGIS LAYER
# =============================================================================
#
# Select and order the columns for the production GeoJSON.
# Column groups:
#
#   IDENTIFIERS
#     h3_id, centroid_lng, centroid_lat
#
#   LABELS (ground truth — for thesis validation, hidden in startup MVP)
#     label, cafe_count
#
#   SCORES (the two independent suitability assessments)
#     ahp_score, ahp_rank, rf_probability, rf_rank
#     rank_diff, combined_score
#
#   CLASSIFICATION
#     suitability_tier
#
#   SHAP ATTRIBUTION (for tooltip explainability)
#     top3_factors
#     shap_base_value
#     shap_{display_name} × 13 features
#
#   GEOMETRY
#     H3 polygon in WGS84

print("Assembling final WebGIS layer columns...")
print()

# Core columns always present
core_cols = [
    'h3_id',
    'centroid_lng',
    'centroid_lat',
    'label',
    'cafe_count',
    'is_validation_cell',   # frontend held-out cell filtering
    'ahp_score',
    'ahp_rank',
    'rf_probability',
    'rf_rank',
    'rank_diff',
    'combined_score',
    'suitability_tier',
    'top3_factors',
    'shap_base_value',
]

# All renamed SHAP columns
renamed_shap_cols = list(shap_rename_map.values())

# Full column list (core + SHAP + geometry)
all_export_cols = core_cols + renamed_shap_cols + ['geometry']

# Keep only columns that exist in the GeoDataFrame
cols_present = [c for c in all_export_cols if c in shap_gdf.columns]
cols_missing  = [c for c in all_export_cols if c not in shap_gdf.columns]

if cols_missing:
    print(f"  WARNING — columns not found (will be skipped): {cols_missing}")

webgis_gdf = shap_gdf[cols_present].copy()

# Round numeric columns for file size efficiency
# AHP/RF scores: 4 decimal places is sufficient
# SHAP values: 6 decimal places preserves enough precision for display
score_cols = ['ahp_score', 'rf_probability', 'combined_score']
shap_value_cols = [c for c in cols_present if c.startswith('shap_')]

for col in score_cols:
    if col in webgis_gdf.columns:
        webgis_gdf[col] = webgis_gdf[col].round(4)

for col in shap_value_cols:
    if col in webgis_gdf.columns:
        webgis_gdf[col] = webgis_gdf[col].round(6)

print(f"WebGIS GeoDataFrame assembled:")
print(f"  Cells:    {len(webgis_gdf)}")
print(f"  Columns:  {len(webgis_gdf.columns)}")
print()
print("Column inventory:")
col_groups = {
    'Identifiers':   ['h3_id', 'centroid_lng', 'centroid_lat'],
    'Ground truth':  ['label', 'cafe_count'],
    'Scores':        ['ahp_score', 'ahp_rank', 'rf_probability',
                      'rf_rank', 'rank_diff', 'combined_score'],
    'Classification':['suitability_tier'],
    'Attribution':   ['top3_factors', 'shap_base_value'] + renamed_shap_cols,
    'Geometry':      ['geometry'],
}
for group, cols in col_groups.items():
    present = [c for c in cols if c in webgis_gdf.columns]
    print(f"  {group:<16}: {len(present)} columns")

Assembling final WebGIS layer columns...

WebGIS GeoDataFrame assembled:
  Cells:    971
  Columns:  29

Column inventory:
  Identifiers     : 3 columns
  Ground truth    : 2 columns
  Scores          : 6 columns
  Classification  : 1 columns
  Attribution     : 15 columns
  Geometry        : 1 columns


# Cell 8

In [ ]:
# =============================================================================
# VALIDATION CHECKS
# =============================================================================
#
# Before saving, verify the GeoJSON is frontend-ready:
#   1. All cells have valid geometries
#   2. No NaN values in score columns
#   3. Suitability tier covers all rows
#   4. Combined score is in [0, 1]
#   5. AHP and RF scores are in expected ranges
#   6. SHAP additivity: base + sum(SHAP) ≈ rf_probability

print("Running WebGIS layer validation checks...")
print()

checks_passed = True

# Check 1 — Valid geometries
n_invalid = webgis_gdf.geometry.isna().sum() + (~webgis_gdf.geometry.is_valid).sum()
check1 = n_invalid == 0
print(f"  [{'PASS' if check1 else 'FAIL'}] Valid geometries: "
      f"{n_invalid} invalid/null geometries")
if not check1:
    checks_passed = False

# Check 2 — No NaN in score columns
nan_scores = webgis_gdf[['ahp_score', 'rf_probability',
                          'combined_score']].isna().sum().sum()
check2 = nan_scores == 0
print(f"  [{'PASS' if check2 else 'FAIL'}] No NaN scores: "
      f"{nan_scores} NaN values in score columns")
if not check2:
    checks_passed = False

# Check 3 — Suitability tier covers all rows
n_tiered = webgis_gdf['suitability_tier'].isin(['HIGH', 'MEDIUM', 'LOW']).sum()
check3 = n_tiered == len(webgis_gdf)
print(f"  [{'PASS' if check3 else 'FAIL'}] Suitability tier complete: "
      f"{n_tiered}/{len(webgis_gdf)} cells classified")
if not check3:
    checks_passed = False

# Check 4 — Combined score in [0, 1]
c_min = webgis_gdf['combined_score'].min()
c_max = webgis_gdf['combined_score'].max()
check4 = (c_min >= 0.0) and (c_max <= 1.0)
print(f"  [{'PASS' if check4 else 'FAIL'}] Combined score in [0,1]: "
      f"min={c_min:.4f}, max={c_max:.4f}")
if not check4:
    checks_passed = False

# Check 5 — AHP and RF score ranges
ahp_ok = (webgis_gdf['ahp_score'].min() >= 0) and \
         (webgis_gdf['ahp_score'].max() <= 1)
rf_ok  = (webgis_gdf['rf_probability'].min() >= 0) and \
         (webgis_gdf['rf_probability'].max() <= 1)
check5 = ahp_ok and rf_ok
print(f"  [{'PASS' if check5 else 'FAIL'}] AHP score range: "
      f"[{webgis_gdf['ahp_score'].min():.4f}, "
      f"{webgis_gdf['ahp_score'].max():.4f}]")
print(f"  [{'PASS' if check5 else 'FAIL'}] RF probability range: "
      f"[{webgis_gdf['rf_probability'].min():.4f}, "
      f"{webgis_gdf['rf_probability'].max():.4f}]")
if not check5:
    checks_passed = False

# Check 6 — SHAP additivity: base + sum(SHAP) ≈ rf_probability
shap_feature_cols_present = [c for c in renamed_shap_cols
                              if c in webgis_gdf.columns]

feature_cols_path = os.path.join(GOLD_PATH, 'gold_feature_cols.pkl')
if os.path.exists(feature_cols_path):
    with open(feature_cols_path, 'rb') as _f:
        FEATURE_COLS_EXPECTED = pickle.load(_f)
    expected_n_shap = len(FEATURE_COLS_EXPECTED)
    actual_n_shap   = len(shap_feature_cols_present)
    col_count_ok    = actual_n_shap == expected_n_shap
    print(f"  [{'PASS' if col_count_ok else 'FAIL'}] SHAP column count: "
          f"{actual_n_shap} present, {expected_n_shap} expected "
          f"(from gold_feature_cols.pkl)")
    if not col_count_ok:
        checks_passed = False
        print(f"    CRITICAL: {expected_n_shap - actual_n_shap} SHAP columns missing.")
        print(f"    Additivity check will not run — fix upstream before proceeding.")
else:
    col_count_ok = True   # fallback: skip assertion if pkl not found
    print(f"  [SKIP] SHAP column count check — gold_feature_cols.pkl not found")

if col_count_ok and 'shap_base_value' in webgis_gdf.columns \
        and len(shap_feature_cols_present) > 0:
    # Use unrounded rf_probability from shap_gdf (pre-rounding source) so that
    # both sides of the comparison are not independently rounded quantities.
    # shap_gdf is still in memory and retains the full-precision values.
    shap_sum      = webgis_gdf[shap_feature_cols_present].sum(axis=1)
    reconstructed = webgis_gdf['shap_base_value'] + shap_sum
    # Align index and get unrounded rf_probability from source GeoDataFrame
    rf_prob_unrounded = shap_gdf.loc[webgis_gdf.index, 'rf_probability'] \
        if 'rf_probability' in shap_gdf.columns \
        else webgis_gdf['rf_probability']
    additivity_error = (reconstructed - rf_prob_unrounded).abs().max()
    check6 = additivity_error <= 0.01
    print(f"  [{'PASS' if check6 else 'WARN'}] SHAP additivity max error: "
          f"{additivity_error:.6f} (threshold: 0.01, vs unrounded rf_probability)")
    if not check6:
        print(f"    NOTE: Error > 0.01 exceeds expected rounding tolerance.")
        print(f"    Verify TreeExplainer model_output='probability' in Notebook 7.")
        checks_passed = False
else:
    print(f"  [SKIP] SHAP additivity check — skipped due to column count failure"
          f" or missing base value")

Running WebGIS layer validation checks...

  [PASS] Valid geometries: 0 invalid/null geometries
  [PASS] No NaN scores: 0 NaN values in score columns
  [PASS] Suitability tier complete: 971/971 cells classified
  [PASS] Combined score in [0,1]: min=0.0149, max=0.9912
  [PASS] AHP score range: [0.1043, 0.8016]
  [PASS] RF probability range: [0.0000, 1.0000]
  [PASS] SHAP column count: 13 present, 13 expected (from gold_feature_cols.pkl)
  [PASS] SHAP additivity max error: 0.004994 (threshold: 0.01, vs unrounded rf_probability)


# Cell 9

In [ ]:
# =============================================================================
# SAVE PRODUCTION WEBGIS GEOJSON
# =============================================================================

print("Saving production WebGIS GeoJSON...")
print()

webgis_path = os.path.join(GOLD_PATH, 'gold_webgis_layer.geojson')
webgis_gdf.to_file(webgis_path, driver='GeoJSON')

# Report file size
file_size_mb = os.path.getsize(webgis_path) / (1024 * 1024)
print(f"  Saved: gold_webgis_layer.geojson")
print(f"  Cells: {len(webgis_gdf)}")
print(f"  Size:  {file_size_mb:.2f} MB")
print()

if file_size_mb > 10:
    print("  NOTE: File exceeds 10 MB. For production WebGIS consider:")
    print("    - Converting to PMTiles or MBTiles for tile-based serving")
    print("    - Using tippecanoe to generate vector tiles")
    print("    - Hosting on a PostGIS + pg_featureserv stack (Phase 3)")
elif file_size_mb > 5:
    print("  NOTE: File is > 5 MB. Suitable for direct Mapbox/Leaflet load")
    print("  at Phase 2 Comune di Milano scale (~970 viable cells).")
else:
    print("  File size is within direct browser-load range for Phase 2.")

Saving production WebGIS GeoJSON...

  Saved: gold_webgis_layer.geojson
  Cells: 971
  Size:  1.16 MB

  File size is within direct browser-load range for Phase 2.


# Cell 10

In [ ]:
# =============================================================================
# SAVE WEBGIS METADATA / LEGEND CONFIG (gold_webgis_summary.json)
# =============================================================================
#
# The frontend reads this JSON to configure the map legend, tooltips,
# colour ramp thresholds, and layer descriptions without hardcoding
# these values in JavaScript.

print("Building WebGIS metadata summary...")
print()

# Load performance metrics from NB6 export
perf_path = os.path.join(GOLD_PATH, 'gold_performance.json')
if os.path.exists(perf_path):
    with open(perf_path, 'r', encoding='utf-8') as _f:
        perf = json.load(_f)
    print(f"  Performance metrics loaded from gold_performance.json")
else:
    perf = {
        "rf_auc_validation":           "see gold_rf_log.txt",
        "rf_cv_auc_mean":              "see gold_rf_log.txt",
        "rf_cv_auc_std":               "see gold_rf_log.txt",
        "spearman_ahp_rf_cell_level":  "see gold_rf_log.txt",
        "spearman_ahp_rf_pvalue":      "see gold_rf_log.txt",
        "note": "gold_performance.json not found — add export block to NB6 Cell 12"
    }
    print(f"  WARNING: gold_performance.json not found — using placeholder values.")
    print(f"  Add the following block to Notebook 6 Cell 12:")
    print(f"    perf = {{")
    print(f"      'rf_auc_validation': round(float(val_auc), 4),")
    print(f"      'rf_cv_auc_mean':    round(float(cv_results['test_roc_auc'].mean()), 4),")
    print(f"      'rf_cv_auc_std':     round(float(cv_results['test_roc_auc'].std()), 4),")
    print(f"      'spearman_ahp_rf_cell_level': round(float(corr), 4),")
    print(f"      'spearman_ahp_rf_pvalue':     round(float(pval), 4),")
    print(f"    }}")
    print(f"    with open(os.path.join(GOLD_PATH, 'gold_performance.json'), 'w') as f:")
    print(f"        json.dump(perf, f, indent=2)")

tier_counts = webgis_gdf['suitability_tier'].value_counts().to_dict()

summary = {
    "project":       "Daqiq — GeoAI Café Site Selection · Milan",
    "phase":         "Phase 2 — Comune di Milano scale",
    "generated":     datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    "study_area":    "Comune di Milano (All 9 Municipi)",
    "h3_resolution": 9,
    "cells": {
        "total_viable":   len(webgis_gdf),
        "positive_label": int(webgis_gdf['label'].sum()),
        "negative_label": int((webgis_gdf['label'] == 0).sum()),
        "positive_rate":  round(float(webgis_gdf['label'].mean()), 4),
    },
    "suitability_tiers": {
        "HIGH":   int(tier_counts.get('HIGH', 0)),
        "MEDIUM": int(tier_counts.get('MEDIUM', 0)),
        "LOW":    int(tier_counts.get('LOW', 0)),
        "thresholds": {
            "combined_high":  COMBINED_HIGH_THRESHOLD,
            "combined_low":   COMBINED_LOW_THRESHOLD,
            "logic":          "HIGH: combined_score >= combined_high; "
                              "MEDIUM: combined_low <= combined_score < combined_high; "
                              "LOW: combined_score < combined_low",
            "informational_only": {
                "ahp_high":  AHP_HIGH_THRESHOLD,
                "ahp_med":   AHP_MED_THRESHOLD,
                "rf_high":   RF_HIGH_THRESHOLD,
                "rf_med":    RF_MED_THRESHOLD,
                "note":      "AHP and RF quantiles are not used in tier assignment. "
                             "Tiers are derived exclusively from combined_score quantiles "
                             "computed on training cells only."
            }
        }
    },
    "score_ranges": {
        "ahp_score":      {"min": round(float(webgis_gdf['ahp_score'].min()), 4),
                           "max": round(float(webgis_gdf['ahp_score'].max()), 4),
                           "mean": round(float(webgis_gdf['ahp_score'].mean()), 4)},
        "rf_probability": {"min": round(float(webgis_gdf['rf_probability'].min()), 4),
                           "max": round(float(webgis_gdf['rf_probability'].max()), 4),
                           "mean": round(float(webgis_gdf['rf_probability'].mean()), 4)},
        "combined_score": {"min": round(float(webgis_gdf['combined_score'].min()), 4),
                           "max": round(float(webgis_gdf['combined_score'].max()), 4),
                           "mean": round(float(webgis_gdf['combined_score'].mean()), 4)},
    },
    "model_performance": perf,
    "features": {
        "count": len(shap_feature_cols_present),
        "shap_columns": shap_feature_cols_present,
        "ahp_clusters": {
            "Accessibility":    {"weight": 0.3764,
                                 "features": ["Metro Access", "Bus/Tram Density",
                                              "Network Centrality"]},
            "Demand Potential": {"weight": 0.2998,
                                 "features": ["Population Density", "Night Light"]},
            "Urban Context":    {"weight": 0.2535,
                                 "features": ["Retail Density", "POI Diversity",
                                              "Pedestrian Street", "Tourist POIs"]},
            "Competition":      {"weight": 0.0703,
                                 "features": ["Market Opportunity"]},
        },
        "rf_only_features": ["Café Density", "Office Density", "University Proximity"],
        "note": "Café Density excluded from AHP (criterion redundancy). "
                "Retained in RF per Han et al. 2024 co-location theory."
    },
    "colour_ramp": {
        "field":       "combined_score",
        "type":        "continuous",
        "method":      "percentile-rank average of ahp_score and rf_probability",
        "note":        "combined_score is uniformly distributed in [0,1]. "
                       "Colour stops are percentile-based, not absolute score values.",
        "stops": [
            [0.00, "#d73027"],
            [0.20, "#f46d43"],
            [0.35, "#fdae61"],
            [0.50, "#a6d96a"],
            [0.65, "#1a9850"],
        ],
        "tier_colours": {
            "HIGH":   "#1a9850",
            "MEDIUM": "#fdae61",
            "LOW":    "#d73027",
        }
    },
    "files": {
        "gold_webgis_layer.geojson": "Production WebGIS layer — all cells, "
                                     "all scores, SHAP vectors, tier",
        "gold_shap_scores.geojson":  "Full Gold SHAP output (includes raw "
                                     "feature col names — not for frontend)",
        "gold_ahp_scores.geojson":   "AHP scores and cluster scores per cell",
        "gold_rf_scores.geojson":    "RF probability scores and ranks",
        "gold_ahp_vs_shap_comparison.csv": "Core thesis comparison table",
        "gold_shap_importance.csv":  "Global SHAP feature importance",
        "gold_rf_model.pkl":         "Trained Random Forest (Phase 2 starting point)",
        "gold_feature_cols.pkl":     "Feature column order for model inference",
    }
}

summary_path = os.path.join(GOLD_PATH, 'gold_webgis_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Summary saved: gold_webgis_summary.json")
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))

Building WebGIS metadata summary...

  Performance metrics loaded from gold_performance.json
Summary saved: gold_webgis_summary.json

{
  "project": "Daqiq — GeoAI Café Site Selection · Milan",
  "phase": "Phase 2 — Comune di Milano scale",
  "generated": "2026-04-26 13:13",
  "study_area": "Comune di Milano (All 9 Municipi)",
  "h3_resolution": 9,
  "cells": {
    "total_viable": 971,
    "positive_label": 345,
    "negative_label": 626,
    "positive_rate": 0.3553
  },
  "suitability_tiers": {
    "HIGH": 359,
    "MEDIUM": 324,
    "LOW": 288,
    "thresholds": {
      "combined_high": 0.6153,
      "combined_low": 0.30476400000000003,
      "logic": "HIGH: combined_score >= combined_high; MEDIUM: combined_low <= combined_score < combined_high; LOW: combined_score < combined_low",
      "informational_only": {
        "ahp_high": 0.5312362685708546,
        "ahp_med": 0.43508378129594105,
        "rf_high": 0.69649,
        "rf_med": 0.04853166666666668,
        "note": "AHP and RF 

# Cell 11

In [ ]:
# =============================================================================
# CELL 11 — PHASE 2 PIPELINE COMPLETION REPORT
# =============================================================================

print("=" * 65)
print("PHASE 2 PIPELINE — COMPLETION REPORT")
print("=" * 65)
print()

# Each tuple: (label, description, canonical output file for that notebook)
notebooks = [
    ("NB1", "Data Ingestion (Bronze)",             "ingestion_log.txt"),
    ("NB2", "Cleaning & Standardization (Silver)", "silver_cleaning_log.txt"),
    ("NB3", "H3 Grid Generation",                  "silver_h3_viable.geojson"),
    ("NB4", "Feature Engineering (Gold)",          "gold_features_normalized.geojson"),
    ("NB5", "AHP Scoring",                         "gold_ahp_scores.geojson"),
    ("NB6", "Random Forest Training",              "gold_rf_scores.geojson"),
    ("NB7", "SHAP + AHP vs SHAP Comparison",       "gold_ahp_vs_shap_comparison.csv"),
    ("NB8", "WebGIS Export",                       "gold_webgis_layer.geojson"),
]

# NB1 output lives in Bronze; NB2–NB3 in Silver; NB4+ in Gold
NB_PATH_MAP = {
    "NB1": BRONZE_PATH,
    "NB2": SILVER_PATH,
    "NB3": SILVER_PATH,
}

all_complete = True
for nb, desc, check_file in notebooks:
    folder = NB_PATH_MAP.get(nb, GOLD_PATH)
    path   = os.path.join(folder, check_file)
    status = "COMPLETE" if os.path.exists(path) else "MISSING OUTPUT"
    if "MISSING" in status:
        all_complete = False
    print(f"  [{status:<16}] {nb}: {desc}")
    print(f"                       checks: {check_file}")

print()

gold_files = [
    'gold_webgis_layer.geojson',
    'gold_webgis_summary.json',
    'gold_shap_scores.geojson',
    'gold_shap_importance.csv',
    'gold_ahp_vs_shap_comparison.csv',
    'gold_shap_bar.png',
    'gold_shap_beeswarm.png',
    'gold_shap_waterfall_top1.png',
    'gold_shap_waterfall_divergent.png',
    'gold_ahp_scores.geojson',
    'gold_rf_scores.geojson',
    'gold_rf_model.pkl',
    'gold_feature_cols.pkl',
    'gold_train_val_idx.pkl',
    'gold_minmax_scaler.pkl',
    'gold_sat_max_train.pkl',
    'gold_features_normalized.geojson',
    'gold_features_raw.geojson',
    'gold_performance.json',
    'gold_roc_curve.png',
    'gold_threshold_sensitivity.csv',   # ← added (NB6 output)
]

print("Gold layer file inventory:")
for fname in gold_files:
    fpath  = os.path.join(GOLD_PATH, fname)
    exists = os.path.exists(fpath)
    size   = f"{os.path.getsize(fpath)/1024:.0f} KB" if exists else "—"
    print(f"  [{'OK' if exists else 'MISSING':>7}] {fname:<50} {size}")

print()
if all_complete:
    print("Phase 2 Comune di Milano scale: COMPLETE")
    print()
    print("Primary thesis deliverables:")
    print("  gold_webgis_layer.geojson       — WebGIS frontend input")
    print("  gold_ahp_vs_shap_comparison.csv — Core thesis comparison table")
    print("  gold_shap_bar.png               — Figure: SHAP feature importance")
    print("  gold_shap_beeswarm.png          — Figure: SHAP beeswarm summary")
    print("  gold_shap_waterfall_*.png       — Figure: cell-level explanations")
    print("  gold_roc_curve.png              — Figure: model ROC curve")
    print()
    print("Next steps: Phase 3 — WebGIS serving layer deployment")
else:
    print("Some pipeline steps appear incomplete.")
    print("Re-run the notebooks marked MISSING OUTPUT before proceeding.")

PHASE 2 PIPELINE — COMPLETION REPORT

  [COMPLETE        ] NB1: Data Ingestion (Bronze)
                       checks: ingestion_log.txt
  [COMPLETE        ] NB2: Cleaning & Standardization (Silver)
                       checks: silver_cleaning_log.txt
  [COMPLETE        ] NB3: H3 Grid Generation
                       checks: silver_h3_viable.geojson
  [COMPLETE        ] NB4: Feature Engineering (Gold)
                       checks: gold_features_normalized.geojson
  [COMPLETE        ] NB5: AHP Scoring
                       checks: gold_ahp_scores.geojson
  [COMPLETE        ] NB6: Random Forest Training
                       checks: gold_rf_scores.geojson
  [COMPLETE        ] NB7: SHAP + AHP vs SHAP Comparison
                       checks: gold_ahp_vs_shap_comparison.csv
  [COMPLETE        ] NB8: WebGIS Export
                       checks: gold_webgis_layer.geojson

Gold layer file inventory:
  [     OK] gold_webgis_layer.geojson                          1189 KB
  [     OK] gold_

# Cell 12

In [ ]:
# =============================================================================
# LOG FILE
# =============================================================================

log_lines = [
    f"Thesis — Phase 2 · WebGIS Export and Preparation Log",
    f"======================================================",
    f"Run date:              {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"",
    f"Input sources:",
    f"  gold_shap_scores.geojson:        {len(shap_gdf)} cells",
    f"  gold_ahp_vs_shap_comparison.csv: {len(comparison_df)} features",
    f"  gold_performance.json:           "
    f"{'loaded' if os.path.exists(perf_path) else 'NOT FOUND — used fallback'}",
    f"  gold_feature_cols.pkl:           "
    f"{'loaded' if os.path.exists(feature_cols_path) else 'NOT FOUND — col check skipped'}",
    f"",
    f"combined_score method: percentile-rank average (AHP + RF)",
    f"  (distribution-agnostic; both methods contribute equally to colour ramp)",
    f"",
    f"WebGIS layer statistics:",
    f"  Cells exported:      {len(webgis_gdf)}",
    f"  Columns:             {len(webgis_gdf.columns)}",
    f"  File size:           {file_size_mb:.2f} MB",
    f"  CRS:                 WGS84 EPSG:4326",
    f"  H3 resolution:       9",
    f"",
    f"Suitability tier distribution:",
    f"  HIGH:   {tier_counts.get('HIGH', 0)} cells",
    f"  MEDIUM: {tier_counts.get('MEDIUM', 0)} cells",
    f"  LOW:    {tier_counts.get('LOW', 0)} cells",
    f"  Thresholds (combined_score quantiles from training cells): "
    f"HIGH>={COMBINED_HIGH_THRESHOLD:.4f}, LOW<={COMBINED_LOW_THRESHOLD:.4f}",
    f"",
    f"Score ranges in exported layer:",
    f"  AHP score:      [{webgis_gdf['ahp_score'].min():.4f}, "
    f"{webgis_gdf['ahp_score'].max():.4f}]",
    f"  RF probability: [{webgis_gdf['rf_probability'].min():.4f}, "
    f"{webgis_gdf['rf_probability'].max():.4f}]",
    f"  Combined score: [{webgis_gdf['combined_score'].min():.4f}, "
    f"{webgis_gdf['combined_score'].max():.4f}]",
    f"",
    f"Validation: {'ALL CHECKS PASSED' if checks_passed else 'SOME CHECKS FAILED'}",
    f"",
    f"Outputs:",
    f"  gold_webgis_layer.geojson",
    f"  gold_webgis_summary.json",
    f"  gold_nb8_log.txt",
]

log_path = os.path.join(GOLD_PATH, 'gold_nb8_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

Thesis — Phase 2 · WebGIS Export and Preparation Log
Run date:              2026-04-26 13:13

Input sources:
  gold_shap_scores.geojson:        971 cells
  gold_ahp_vs_shap_comparison.csv: 13 features
  gold_performance.json:           loaded
  gold_feature_cols.pkl:           loaded

combined_score method: percentile-rank average (AHP + RF)
  (distribution-agnostic; both methods contribute equally to colour ramp)

WebGIS layer statistics:
  Cells exported:      971
  Columns:             29
  File size:           1.16 MB
  CRS:                 WGS84 EPSG:4326
  H3 resolution:       9

Suitability tier distribution:
  HIGH:   359 cells
  MEDIUM: 324 cells
  LOW:    288 cells
  Thresholds (combined_score quantiles from training cells): HIGH>=0.6153, LOW<=0.3048

Score ranges in exported layer:
  AHP score:      [0.1043, 0.8016]
  RF probability: [0.0000, 1.0000]
  Combined score: [0.0149, 0.9912]

Validation: ALL CHECKS PASSED

Outputs:
  gold_webgis_layer.geojson
  gold_webgis_summary.